# Causal Identification under Network Interference

## Business Question

本项目研究在网络干扰（network interference）存在的情况下，
员工之间的 AI 工具接受情况是否会影响个体结果。

具体问题：

对于员工 i，在自身 treatment 状态和合作网络 exposure 状态确定时，
对应的因果效应是否能够被识别（identified）？

# Setting


## Network Structure

每个员工表示为网络中的一个节点（node）。

如果两个员工之间存在合作关系，则二者之间存在一条边（edge）。

本实验使用如下合作网络：

$$
G=(V,E)
$$ 


## Treatment Assignment

采用 D2 fixed-size assignment design。

共有 N 个员工，其中固定 m 个员工接受 AI training。

因此：

$$
\sum_i T_i=m
$$


所有满足该条件的 treatment assignment 具有相同概率。


## Exposure Mapping

对于员工 i：

$$
K_i(t)=\sum_{j\in N(i)}T_j
$$

其中：

- $N(i)$：员工 i 的合作邻居集合
- $K_i(t)$：接受 AI 的邻居数量


采用 strict majority exposure：

$$
S_i(t)=1\{K_i(t)>\lfloor d_i/2\rfloor\}
$$


其中：

$$
d_i=|N(i)|
$$


## Target

关注 joint arm：

$$
(T_i,S_i)
$$

并研究：

$$
\mu_i(t,s)
$$

是否可以被 point identified。


## Assumptions

本实验基于以下 assumptions：

1. Consistency

2. Full-vector randomization

3. Positive support condition

在本 demo 中，默认上述 assumptions 已经满足。

因此，rule.py 不负责检验这些 assumptions，
而是在 assumptions 已确认成立的情况下，
进一步判断对应 joint arm 是否具有 positive support。

这里设置一个小规模网络用于 demonstration。

共有4个员工节点，其中2个员工接受 AI training。

研究员工1在不同 joint arm 下的 identification 情况。

In [21]:
from support import calculate_support
from rule import identify_arm


# Define network graph

graph = {
    1:[2,3,4],
    2:[1],
    3:[1],
    4:[1]
}


# Fixed number of treated individuals

m = 2


# Target individual

node = 1

## Support 的含义

代码通过枚举所有满足 D2 assignment design 的 treatment assignment，
计算每一种情况下员工 i 对应的 joint arm：

$$
(T_i,S_i)
$$

出现的频率，从而得到：

$$
\pi_i(t,s)=P(T_i=t,S_i=s)
$$


对于小规模网络，该枚举方法与理论上的组合概率计算结果一致。

如果：

$$
\pi_i(t,s)>0
$$

说明该 joint arm 在当前实验设计下可以出现。


如果：

$$
\pi_i(t,s)=0
$$

说明该状态在当前设计下不存在。

In [22]:
pi = calculate_support(
    graph,
    m,
    node
)


print("Support probability:")
pi

Support probability:


{(1, 1): 0.0, (1, 0): 0.5, (0, 1): 0.5, (0, 0): 0.0}

## POINT Identification

当：

$$
\pi_i(t,s)>0
$$

并且 assumptions 已经满足时：

- Randomization 保证 treatment assignment 与潜在结果独立；
- Consistency 将观测结果与潜在结果连接；
- Positive support 保证目标状态存在。


因此：

$$
\mu_i(t,s)
$$

可以由：

$$
E[Y_i^{obs}|A,X,T_i=t,S_i=s]
$$

唯一确定。

此时返回：

POINT

### POINT Identification Example

在当前 assignment design 下：

$$
m=2
$$

计算得到的 support 中：

$$
\pi_i(1,0)>0
$$

因此 joint arm：

$$
(T_i,S_i)=(1,0)
$$

满足 positive support condition。

调用 identification rule 后，该 causal query 被判定为：

POINT

In [23]:
result = identify_arm(
    pi,
    1,
    0
)


result

{'status': 'POINT',
 'arm': (1, 0),
 'functional': 'mu_i(1,0)=E[Y_i_obs | A,X,T_i=1,S_i=0]'}

## Zero Support Case

为了展示 zero-support 情况，
这里设置固定只有一个员工接受 AI training：

$$
m=1
$$


对于员工 1：

如果：

$$
T_1=1
$$

由于剩余 treatment 名额为 0，
其合作邻居无法达到 strict majority exposure。


因此：

$$
\pi_1(1,1)=0
$$


在这种情况下，该 joint arm 在当前 assignment design 下无法出现。


因此：

$$
\pi_i(t,s)=0
$$

时，causal query cannot be identified。


Identification result:

NOT IDENTIFIED

In [24]:
# Generate a zero-support example

zero_pi = calculate_support(
    graph,
    1,
    node
)


zero_result = identify_arm(
    zero_pi,
    1,
    1
)


zero_result

{'status': 'NOT IDENTIFIED', 'reason': 'Zero support for this arm'}

# Conclusion

本 demo 展示完整的 causal identification pipeline：

$$
\text{Business Question}
\rightarrow
\text{Setting}
\rightarrow
\text{Support}
\rightarrow
\text{Identification Rule}
\rightarrow
\text{Identification Result}
$$


其中：

- support.py 负责计算：

$$
\pi_i(t,s)=P(T_i=t,S_i=s)
$$


- rule.py 根据 support condition 判断：

$$
\pi_i(t,s)>0
$$

时返回：

POINT


$$
\pi_i(t,s)=0
$$

时返回：

NOT IDENTIFIED


该流程展示了在 network interference 场景下，
如何基于 assignment mechanism 和 exposure mapping
进行 causal identification。